# Derive Apple Log 2 → Apple Log Matrix 

Transforming Apple Log 2 to Apple Log involves a change of primaries with a 3x3 matrix, but for such a transformation to be valid, the image data to which the matrix is applied must be linear. Such linearization is outside the scope of this notebook, but linearization of Apple Log data

- is built into the Colour package
- is built into OpenColorIO as of OCIO 2.4.3

The same OETF and inverse OETF are used for both Apple Log and Apple Log 2.

Documentation of Apple Log is available [here](https://download.developer.apple.com/Developer_Tools/Apple_Log_profile/Apple_Log_Profile_WhitePaper_Sept2023_V1.1.pdf) but that link is behind an Apple Developer paywall. Searching the web for "Apple Log Profile White Paper September 2023" might uncover unauthorized copies.

Documentation of Apple Log 2 is available [here](https://download.developer.apple.com/Developer_Tools/Apple_Log_2/Apple_Log_2_White_Paper.pdf) but again, it's behind a developer paywall. Searching for "Apple Log 2 White Paper September 2025" might be productive.

## General strategy

First we use the procedures laid out in SMPTE RP 177 §3.3.1–3.3.7 to derive the NPM [Normalized Primary Matrix] for Apple Log, *i.e.* the matrix that takes an Apple Log RGB value and returns a CIE XYZ tristimulus value (to be pedantic, a CIE 1931 2º Standard Observer tristimulus value).

We build this up as a series of functions culminating in a one-line call supplying primary and white point chromaticities and returning the corresponding NPM.

We call that function, then verify our results against the reference implementation in Colour to verify that the functions are together producing a correct result.

Then we call the function again with the primary and white point chromaticities of the Apple Log 2 encoding.

Finally, using the procedure given in SMPTE RP 177 §4.2.1-4.2.3, we multiply the inverse of the Apple Log NPM (on the left) and the Apple Log NPM (on the right) to get a matrix that takes linear Apple Log 2 RGB and produces linear Apple Log RGB.

Note:
- RGB is normalized such that **reference white maps from R=G=B=1** and has **Y = 1.0** as required by RP 177.

## Input chromaticities for Apple Log are those of ITU-R BT.2020

CIE 1931 (x, y) chromaticities:

- **Red**: (0.708, 0.292)  
- **Green**: (0.170, 0.797)  
- **Blue**: (0.131, 0.046)  
- **Reference white (D65)**: (0.3127, 0.3290)

We will use these values directly as the input to RP 177 §3.3.1.

In [1]:
import numpy as np

# ---- Documented Apple Log chromaticities (CIE 1931 x,y) ----
xR_AL, yR_AL = 0.708, 0.292
xG_AL, yG_AL = 0.170, 0.797
xB_AL, yB_AL = 0.131, 0.046

# Reference white D65 for Apple Log
xW_AL, yW_AL = 0.3127, 0.3290


## SMPTE RP 177 §3.3 step-by-step

We follow RP 177 exactly:

- **§3.3.2**: compute z = 1 − (x + y) for each primary and the white.
- **§3.3.3**: form matrix **P** from the (x, y, z) of primaries; form **W** vector from the white, normalized so that **Y = 1.0**.
- **§3.3.4**: compute scaling coefficients **Cᵢ** as `P⁻¹ · W`.
- **§3.3.5**: form diagonal matrix **C** from Cᵢ.
- **§3.3.6**: compute **NPM = P · C**.
- **§3.3.7**: NPM is the RGB→XYZ matrix for linear signals.

In [2]:
def xy_to_xyz_row(x: float, y: float) -> tuple[float, float, float]:
    """RP 177 §3.3.2: given x,y, compute z."""
    z = 1.0 - (x + y)
    return (x, y, z)


In [3]:
type primary_xy_chromaticities = tuple[tuple[float, float],
                                       tuple[float, float],
                                       tuple[float, float]]
type white_point_chromaticities = tuple[float, float]

def P_and_W_for_chromaticities(primaries: primary_xy_chromaticities,
                               white_point: white_point_chromaticities) -> tuple[np.ndarray, np.ndarray]:
    """§3.3.3: form P matrix and W vector"""
    (xR, yR), (xG, yG), (xB, yB) = primaries
    (xW, yW) = white_point
    zR = xy_to_xyz_row(xR, yR)[2]
    zG = xy_to_xyz_row(xG, yG)[2]
    zB = xy_to_xyz_row(xB, yB)[2]
    mP = np.array([
        [xR, xG, xB],
        [yR, yG, yB],
        [zR, zG, zB]
        ], dtype=np.float64)
    zW = xy_to_xyz_row(xW, yW)[2]
    vW = np.array([[xW / yW], [1.0], [zW / yW]])
    return mP, vW


In [4]:
def scaling_coefficients(mP: np.ndarray, vW: np.ndarray) -> np.ndarray:
    """# §3.3.4: compute scaling coefficients C_i = P^{-1} · W"""
    mP_inv = np.linalg.inv(mP)
    C_vec = mP_inv @ vW   # shape (3,1): [C_R, C_G, C_B]^T

    C_R, C_G, C_B = C_vec.flatten().tolist()
    return np.array([C_R, C_G, C_B])


In [5]:
def diag_matrix_C(diag_elements: np.ndarray) -> np.ndarray:
    """# §3.3.5: diagonal matrix C from coefficients"""
    return np.diag(diag_elements)

In [6]:
def NPM_from_chromaticities(primaries: primary_xy_chromaticities,
                            white_point: white_point_chromaticities) -> np.ndarray:
    mP, vW = P_and_W_for_chromaticities(primaries, white_point)
    vC = scaling_coefficients(mP, vW)
    mC = diag_matrix_C(vC)
    # §3.3.6: NPM = P · C
    npm = mP @ mC
    return npm


In [7]:
def print_matrix(m: np.ndarray, desc: str) -> None:
    """Display matrix with high precision and with 4-decimal rounding (RP 177 §3.2 output convention)"""
    np.set_printoptions(precision=12, suppress=False)

    print(f"{desc} (high precision):")
    print(m)

    print(f"\n{desc} (rounded to 4 decimals):")
    print(np.round(m, 4))
    

In [8]:
npm_AL = NPM_from_chromaticities(((xR_AL, yR_AL), (xG_AL, yG_AL), (xB_AL, yB_AL)), (xW_AL, yW_AL))
print_matrix(npm_AL, 'NPM')

NPM (high precision):
[[0.636958048301 0.144616903586 0.168880975164]
 [0.262700212011 0.677998071519 0.05930171647 ]
 [0.             0.028072693049 1.060985057711]]

NPM (rounded to 4 decimals):
[[0.637  0.1446 0.1689]
 [0.2627 0.678  0.0593]
 [0.     0.0281 1.061 ]]


## Sanity checks

1) **White check** (RP 177 normalization): for RGB = [1,1,1]ᵀ, the resulting XYZ should correspond to the white chromaticity with Y=1.

2) **Column meaning**: columns of NPM are the XYZ tristimulus values of the normalized R, G, B primaries (as scaled by Cᵢ).

In [9]:
def test_white_as_expected(npm: np.ndarray, white_point: white_point_chromaticities) -> None:
    """White check: XYZ_white = NPM @ [1,1,1]^T should equal [xW/yW, 1, zW/yW]^T"""
    xW, yW = white_point
    xW, yW, zW = xy_to_xyz_row(xW, yW)
    vW = np.array([[xW/yW], [1.0], [zW/yW]])
    rgb_white = np.array([[1.0],[1.0],[1.0]], dtype=np.float64)
    XYZ_from_matrix = npm @ rgb_white

    print("XYZ from NPM @ [1,1,1]:\n", XYZ_from_matrix.flatten())
    print("Target W vector:       \n", vW.flatten())

    print("\nDifference:")
    print((XYZ_from_matrix - vW).flatten())
    

In [10]:
test_white_as_expected(npm_AL, (xW_AL, yW_AL))

XYZ from NPM @ [1,1,1]:
 [0.950455927052 1.             1.08905775076 ]
Target W vector:       
 [0.950455927052 1.             1.08905775076 ]

Difference:
[-2.22044604925e-16  0.00000000000e+00  0.00000000000e+00]


In [11]:
inv_npm_AL = np.linalg.inv(npm_AL)

## Result

The **NPM** computed above is the matrix to take linearized 'Apple Log' and turn it into CIE XYZ,
as obtained directly by the RP 177 procedure using the Apple Log primary and white chromaticities.

## Verification Against Colour Reference Values

This checks the derived Apple Log matrices against Colour's built-in reference for `ITU-R BT.2020`, since those are the primaries Apple uses for Apple Log (but **not** for Apple Log 2).

In [12]:
from colour.models import RGB_COLOURSPACES

def check_against_colour(npm: np.ndarray, npm_inv: np.ndarray, colourspace: str) -> None:
    cs = RGB_COLOURSPACES[colourspace]
    M_ref = np.asarray(cs.matrix_RGB_to_XYZ, dtype=np.float64)
    M_inv_ref = np.asarray(cs.matrix_XYZ_to_RGB, dtype=np.float64)

    delta_M = npm - M_ref
    delta_M_inv = npm_inv - M_inv_ref

    max_abs_M = np.max(np.abs(delta_M))
    max_abs_M_inv = np.max(np.abs(delta_M_inv))

    print("Colour reference colourspace:", cs.name)
    print("Max abs diff (RGB->XYZ):", max_abs_M)
    print("Max abs diff (XYZ->RGB):", max_abs_M_inv)

    tolerance = 1e-12
    assert np.allclose(npm, M_ref, atol=tolerance, rtol=0.0), "NPM does not match Colour reference within tolerance."
    assert np.allclose(npm_inv, M_inv_ref, atol=tolerance, rtol=0.0), "NPM_inv does not match Colour reference within tolerance."

    print(f"PASS: Both matrices match Colour reference within atol={tolerance}.")

In [13]:
check_against_colour(npm_AL, inv_npm_AL, "ITU-R BT.2020")

Colour reference colourspace: ITU-R BT.2020
Max abs diff (RGB->XYZ): 4.4408920985e-16
Max abs diff (XYZ->RGB): 3.33066907388e-16
PASS: Both matrices match Colour reference within atol=1e-12.


In [14]:
# ---- Documented Apple Log 2 chromaticities (CIE 1931 x,y) ----
xR_AL2, yR_AL2 = 0.725, 0.301
xG_AL2, yG_AL2 = 0.221, 0.814
xB_AL2, yB_AL2 = 0.068, -0.076

# Reference white D65 for Apple Log
xW_AL2, yW_AL2 = 0.3127, 0.3290


In [15]:
npm_AL2 = NPM_from_chromaticities(((xR_AL2, yR_AL2), (xG_AL2, yG_AL2), (xB_AL2, yB_AL2)), (xW_AL2, yW_AL2))
inv_npm_AL2 = np.linalg.inv(npm_AL2)
al2_to_al = inv_npm_AL @ npm_AL2
print_matrix(al2_to_al, "Apple Log 2 to Apple Log")

Apple Log 2 to Apple Log (high precision):
[[ 1.028100938166  0.098978817671 -0.127079755836]
 [ 0.00252034193   1.170849643324 -0.173369985253]
 [-0.022087573248 -0.064050383441  1.086137956689]]

Apple Log 2 to Apple Log (rounded to 4 decimals):
[[ 1.0281  0.099  -0.1271]
 [ 0.0025  1.1708 -0.1734]
 [-0.0221 -0.0641  1.0861]]


In [16]:
def ocio_style_matrix_transform(m: np.ndarray) -> str:
    result = np.zeros([4, 4])
    result[:3, :3] = m
    result[3, 3] = 1.0
    flat_m = result.flatten()
    ocio_style = ', '.join([str(e) for e in flat_m])
    return f"!<MatrixTransform> {{matrix: [{ocio_style}]}}"


In [17]:
ocio_style_matrix_transform(al2_to_al)

'!<MatrixTransform> {matrix: [1.02810093817, 0.0989788176707, -0.127079755836, 0.0, 0.00252034192972, 1.17084964332, -0.173369985253, 0.0, -0.0220875732483, -0.0640503834411, 1.08613795669, 0.0, 0.0, 0.0, 0.0, 1.0]}'